In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [ ]:
sns.set_theme(style='whitegrid',context='notebook',palette='colorblind')
SELECTED = CONFIG['models']
ORDER = ['empirical_complete'] + SELECTED
PALETTE = dict(zip(ORDER,sns.color_palette('colorblind',len(ORDER))))
DASHES = dict(zip(ORDER,['',(4,2),(1,1),(6,2,1,2),(3,1,1,1),(8,2)]))
MARKERS = dict(zip(ORDER,['o','s','^','D','v','P']))
def save(fig,name):
    fig.tight_layout()
    fig.savefig(OUTPUT/(name+'.png'),dpi=180,bbox_inches='tight')
    fig.savefig(OUTPUT/(name+'.pdf'),bbox_inches='tight')
    plt.show()
    plt.close(fig)
def lines(data,x,y,ax):
    sns.lineplot(data=data,x=x,y=y,hue='source',style='source',hue_order=ORDER,style_order=ORDER,
                 palette=PALETTE,dashes=DASHES,markers=MARKERS,markersize=3,errorbar=None,ax=ax)
def legend(ax):
    ax.legend(fontsize=7,ncol=2)
def cdf_lines(data,x,y,ax):
    # duplicate x values draw vertical jumps; no averaging of left/right ECDF limits
    for source in ORDER:
        d = data.loc[data.source.eq(source)].sort_values(x,kind='stable')
        sns.lineplot(data=d,x=x,y=y,estimator=None,sort=False,color=PALETTE[source],
                     linestyle='-' if not DASHES[source] else (0,DASHES[source]),label=source,ax=ax)
    ax.set_ylim(-.02,1.02)

In [ ]:
pmf = pd.read_csv(INPUT/'annual_rounded_pmf.csv')
survival = pd.read_csv(INPUT/'zero_year_survival.csv')
# load only through the traced simulation dependency
panel = pd.read_csv(INPUT/'simulation/fit/data/panel.csv',index_col='dblp_id')
Q = panel.to_numpy(dtype=float).T
sources = {'empirical_complete':Q[:,np.isfinite(Q).all(axis=0)],
           **{tag:np.load(INPUT/'simulation'/tag/'trajectories.npy') for tag in SELECTED}}

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(12,4))
lines(pmf,'count','probability',axes[0])
axes[0].set_yscale('log')
axes[0].set_xticks([0,5,10,15,20],['0','5','10','15','20+'])
axes[0].set(xlabel='Rounded annual output',ylabel='Probability',title='Annual productivity PMF; overflow shown')
lines(survival,'zero_years','survival',axes[1])
axes[1].set_yscale('log')
axes[1].set(xlabel='Zero years k',ylabel='P(number of zero years ≥ k)',title='Zero-year survival')
legend(axes[0])
axes[1].legend().remove()
save(fig,'annual_pmf_and_zero_survival')

In [ ]:
# initialization uses all available first years
fig,axes = plt.subplots(1,2,figsize=(12,4))
initial = Q[0,np.isfinite(Q[0])]
lo,hi = 0,np.quantile(initial,.995)
edges = np.linspace(lo,hi,51)
for source,array in sources.items():
    values = array[0]
    count,_ = np.histogram(values,bins=edges)
    centers = (edges[:-1]+edges[1:])/2
    sns.lineplot(x=centers,y=count/(len(values)*np.diff(edges)),color=PALETTE[source],
                 linestyle='-' if not DASHES[source] else (0,DASHES[source]),ax=axes[0],label=source)
# available initial sample is shown separately from survivor initial distribution
count,_ = np.histogram(initial,bins=edges)
axes[0].step((edges[:-1]+edges[1:])/2,count/(len(initial)*np.diff(edges)),where='mid',color='.15',ls=':',label='empirical_available')
grid = np.linspace(lo,hi,400)
axes[1].plot(grid,stats.expon.pdf(grid,scale=initial.mean()),color='.2',ls='--',label='Exponential fit to all q0')
axes[1].step((edges[:-1]+edges[1:])/2,count/(len(initial)*np.diff(edges)),where='mid',label='empirical_available')
for ax in axes:
    ax.set(xlabel='Initial productivity q0',ylabel='Density')
    ax.set_yscale('log')
    legend(ax)
fig.suptitle('First-year productivity; central 99.5% display, full-N density')
save(fig,'initial_productivity')

In [ ]:
rows = []
for source,array in sources.items():
    for value in np.log1p(array[:6].sum(axis=0)):rows.append({'source':source,'log1p_total':value})
fig,ax = plt.subplots(figsize=(10,4))
sns.violinplot(data=pd.DataFrame(rows),x='source',y='log1p_total',order=ORDER,hue='source',palette=PALETTE,
               inner='quart',cut=0,density_norm='area',legend=False,ax=ax)
ax.set(xlabel='',ylabel='log1p(total through year 5)',title='Early-career cumulative distribution; AR(6,6) empirical prefix')
ax.tick_params(axis='x',rotation=20)
save(fig,'early_cumulative_violin')

In [ ]:
rng = np.random.default_rng(SEED)
fig,axes = plt.subplots(2,3,figsize=(13,7),sharex=True)
for ax,(source,array) in zip(axes.flat,sources.items()):
    indices = rng.choice(array.shape[1],size=min(18,array.shape[1]),replace=False)
    # representative paths stay in the figure; no trajectory CSV written
    for index in indices:sns.lineplot(x=np.arange(Y+1),y=array[:,index],color=PALETTE[source],alpha=.15,lw=.8,ax=ax)
    sns.lineplot(x=np.arange(Y+1),y=array.mean(axis=1),color=PALETTE[source],lw=2,ax=ax)
    ax.axvline(5.5,color='.6',ls=':',lw=1)
    ax.set_yscale('symlog',linthresh=1)
    ax.set(title=source,xlabel='Model year',ylabel='Annual productivity')
fig.suptitle('Seed-63 sample paths and population mean; AR(6,6) generated from year 6')
save(fig,'representative_trajectories')